# Train the Shelfora shelf detector
Runtime → Change runtime type → **T4 GPU**. Stage 1 learns products on crowded shelves from SKU-110K; stage 2 teaches *product vs empty gap* with Roboflow / Kaggle data and frames from your own cameras.

In [ ]:
!git clone https://github.com/YOUR-USER/shelfora-vision.git || true
%cd shelfora-vision
!pip -q install -r requirements.txt

## Stage 1: SKU-110K base model
The first run downloads SKU-110K (~13.6 GB, a few minutes on Colab). `--fraction 0.3` trains on 30% of it to fit a free session; use 1.0 for the best model.

In [ ]:
!python scripts/train.py --stage base --weights yolo11s.pt --epochs 30 --fraction 0.3 --device 0

## Kaggle: Grocery Items, Multi-class Object Detection
This set names each item, so the model learns *which* product it sees, not just that something is there.
1. On kaggle.com, open the competition and accept its rules (otherwise the download returns 403).
2. Account → Create New API Token, then upload the `kaggle.json` it gives you when the cell below asks.

In [ ]:
from google.colab import files
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    up = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb').write(up['kaggle.json'])
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
!python scripts/get_data.py kaggle-comp --competition grocery-items-multi-class-object-detection

In [ ]:
# Combine with any other multi-class sets (e.g. a Roboflow grocery project) and train
!python scripts/get_data.py merge datasets/kc-grocery-items-multi-class-object-detection* --keep-classes --name grocery
!python scripts/train.py --stage grocery --weights runs/shelfora/base/weights/best.pt --epochs 80 --device 0

In [ ]:
# Link the model's classes to your products, then download both files
!python scripts/map_classes.py --model models/shelfora-grocery.pt --write
files.download('models/shelfora-grocery.pt'); files.download('config/store.yaml')

## Stage 2 data: Roboflow, Kaggle and your store
Pick an empty-shelf / retail-shelf detection project on universe.roboflow.com and copy its workspace, project and version. Labels containing words like *empty*, *gap* or *void* become class `empty`; everything else becomes `product`.

In [ ]:
ROBOFLOW_KEY = ""  # roboflow.com → Settings → API keys
# Empty-gap datasets (CC BY 4.0) and Indian retail shelves — see TRAINING.md
!python scripts/get_data.py roboflow --api-key $ROBOFLOW_KEY --workspace dsjourney --project empty-spaces-detection-in-shelf-data --version 2
!python scripts/get_data.py roboflow --api-key $ROBOFLOW_KEY --workspace area-mcjd4 --project empty-spaces-detection-in-shelf-data-5ddld --version 1
!python scripts/get_data.py roboflow --api-key $ROBOFLOW_KEY --workspace nmims-b6ta9 --project retail-store-indian --version 1

In [ ]:
# Optional Kaggle source (upload kaggle.json to ~/.kaggle first)
# !python scripts/get_data.py kaggle --dataset owner/dataset-name
# !python scripts/get_data.py convert --src datasets/kg-dataset-name --format voc

In [ ]:
# Your own store frames, labelled in Roboflow and exported back
# !python scripts/get_data.py roboflow --api-key $ROBOFLOW_KEY --workspace YOUR_WS --project dgi-store-shelves --version 1
!python scripts/get_data.py merge datasets/rf-*

## Stage 2: fine-tune

In [ ]:
!python scripts/train.py --stage finetune --weights runs/shelfora/base/weights/best.pt --epochs 60 --device 0

In [ ]:
from ultralytics import YOLO
from IPython.display import Image
res = YOLO('models/shelfora-shelf.pt').predict('datasets/shelfora/val/images', max_det=1000, save=True, imgsz=1024)
Image(filename=res[0].save_dir + '/' + __import__('os').listdir(res[0].save_dir)[0])

In [ ]:
from google.colab import files
files.download('models/shelfora-shelf.pt')